# 🏗️ Notebook 04 — Two-Tower Neural Recommendation System

> **Project:** Context-Aware Neural Recommendation Engine  
> **Phase:** Core Deep Learning — Retrieval Model  
> **Author:** Internship Engineering Team  
> **Status:** 🟢 Active Development

---

## 🎯 What Will You Learn in This Notebook?

| Topic | Description |
|---|---|
| Two-Tower Architecture | How recommendation systems learn user & item representations |
| Embeddings | What they are, why they matter, how they power modern AI |
| TF Datasets | How to prepare interaction data for training |
| TFRS Retrieval | How to train a retrieval model using TensorFlow Recommenders |
| Generating Recommendations | Getting Top-K items for any user |

---

## 📚 Section 1 — What is a Two-Tower Recommendation System?

### 🌍 The Big Picture: How Do Netflix, YouTube, and Amazon Recommend Things?

Every time you open Netflix, it doesn't randomly show you movies. It runs a **Recommendation System** — a machine learning system trained to predict: **"What will THIS user most likely enjoy?"**

Modern recommendation systems work in **two stages**:

```
Stage 1: RETRIEVAL (What we build here)
   └── From millions of items → shortlist Top 100-500 candidates
   └── Fast, approximate, uses embeddings

Stage 2: RANKING (Next notebook)
   └── From Top 500 candidates → final ordered Top 10
   └── Slower, more precise, uses richer features
```

### 🏗️ What is the Two-Tower Architecture?

The **Two-Tower model** (also called Dual Encoder) is the industry-standard retrieval architecture used by:
- **Google** — for YouTube recommendations
- **Meta** — for Facebook/Instagram feeds
- **Spotify** — for music discovery
- **Amazon** — for product recommendations

```
                    ┌─────────────────────────────────────┐
                    │       TWO-TOWER ARCHITECTURE        │
                    └─────────────────────────────────────┘

  USER DATA                               ITEM DATA
 ┌──────────┐                           ┌──────────┐
 │ user_id  │                           │ item_id  │
 │ age      │                           │ category │
 │ location │                           │ price    │
 └────┬─────┘                           └────┬─────┘
      │                                      │
      ▼                                      ▼
 ┌──────────┐                           ┌──────────┐
 │  QUERY   │                           │CANDIDATE │
 │  TOWER   │                           │  TOWER   │
 │(User NN) │                           │(Item NN) │
 └────┬─────┘                           └────┬─────┘
      │                                      │
      ▼                                      ▼
 [0.2, 0.8, 0.1, ...]              [0.3, 0.7, 0.2, ...]
   User Embedding                    Item Embedding
      │                                      │
      └──────────────┬───────────────────────┘
                     ▼
              Dot Product / Cosine Similarity
                     │
                     ▼
              SIMILARITY SCORE
              (Higher = Better Match)
```

### 🧠 What are Embeddings?

An **embedding** is a list of numbers (a vector) that represents the "meaning" or "characteristics" of something.

**Example:** Imagine User A likes action movies. Their embedding might look like:
```
User A embedding: [0.9, 0.1, 0.8, 0.2]  ← loves action, dislikes romance
User B embedding: [0.8, 0.2, 0.9, 0.1]  ← similar to User A!
User C embedding: [0.1, 0.9, 0.2, 0.8]  ← opposite to User A
```

**The magic:** The model LEARNS these numbers automatically from interaction data. We never hand-code them.

**Key insight:** If a user embedding is SIMILAR to an item embedding → that user will likely enjoy that item.

## 📦 Section 2 — Import Libraries

In [1]:
# ─────────────────────────────────────────────────────────────
# STANDARD LIBRARIES
# ─────────────────────────────────────────────────────────────
import os
import warnings
import numpy as np
import pandas as pd

# ─────────────────────────────────────────────────────────────
# TENSORFLOW — Core deep learning framework
# We use TensorFlow to build neural networks
# ─────────────────────────────────────────────────────────────
import tensorflow as tf
print(f"✅ TensorFlow version: {tf.__version__}")

# ─────────────────────────────────────────────────────────────
# TENSORFLOW RECOMMENDERS (TFRS)
# Built on top of TensorFlow specifically for recommendation systems
# Handles retrieval tasks, ranking tasks, and evaluation metrics
# ─────────────────────────────────────────────────────────────
import tensorflow_recommenders as tfrs
print(f"✅ TensorFlow Recommenders version: {tfrs.__version__}")

# ─────────────────────────────────────────────────────────────
# SUPPRESS UNNECESSARY WARNINGS (common in ML projects)
# ─────────────────────────────────────────────────────────────
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # Suppress TF info logs

print("\n🚀 All libraries loaded successfully!")


✅ TensorFlow version: 2.21.0
✅ TensorFlow Recommenders version: v0.7.7

🚀 All libraries loaded successfully!


## 📂 Section 3 — Load Feature Datasets

We load the three datasets generated by our preprocessing and feature engineering pipelines.

| Dataset | Purpose | Key Columns |
|---|---|---|
| `user_features.csv` | Describes each user | user_id, age, location, segment |
| `item_features.csv` | Describes each product | item_id, category, price, brand |
| `interaction_features.csv` | Records user-item interactions | user_id, item_id, rating, timestamp |

In [23]:
# ─────────────────────────────────────────────────────────────
# DEFINE DATA PATHS
# Using relative paths so the project works on any machine
# ─────────────────────────────────────────────────────────────
DATA_DIR = os.path.join("..", "data", "processed")

USER_FEATURES_PATH        = os.path.join(DATA_DIR, "user_features.csv")
ITEM_FEATURES_PATH        = os.path.join(DATA_DIR, "item_features.csv")
ARTICLES_FEATURES_PATH    = os.path.join(DATA_DIR, "articles_cleaned.csv")
INTERACTION_FEATURES_PATH = os.path.join(DATA_DIR, "interaction_features.csv")

# ─────────────────────────────────────────────────────────────
# LOAD DATASETS
# For fast experimentation we load the full file once, then keep only a
# smaller sample. Full recommendation datasets are very large, and training
# on every row can take hours on a laptop. Teams usually prototype on a
# subset first to validate the architecture, debug the pipeline, and get
# quick feedback before moving to production-scale training.
# ─────────────────────────────────────────────────────────────
print("📂 Loading datasets...")

user_df        = pd.read_csv(USER_FEATURES_PATH)
item_df        = pd.read_csv(ITEM_FEATURES_PATH)
articles_df    = pd.read_csv(ARTICLES_FEATURES_PATH)
interaction_df = pd.read_csv(INTERACTION_FEATURES_PATH)

# article_id must be a string so it matches the model outputs
articles_df["article_id"] = articles_df["article_id"].astype(str)

# Keep a smaller subset for faster local experimentation
interaction_df = interaction_df.sample(20000, random_state=42)

print(f"✅ Users loaded:        {user_df.shape[0]:,} rows, {user_df.shape[1]} columns")
print(f"✅ Items loaded:        {item_df.shape[0]:,} rows, {item_df.shape[1]} columns")
print(f"✅ Articles loaded:     {articles_df.shape[0]:,} rows, {articles_df.shape[1]} columns")
print(f"✅ Interactions sampled: {interaction_df.shape[0]:,} rows, {interaction_df.shape[1]} columns")

📂 Loading datasets...
✅ Users loaded:        1,371,980 rows, 16 columns
✅ Items loaded:        105,542 rows, 33 columns
✅ Articles loaded:     105,542 rows, 25 columns
✅ Interactions sampled: 20,000 rows, 13 columns


In [3]:
# ─────────────────────────────────────────────────────────────
# QUICK DATASET EXPLORATION
# Always inspect your data before modeling
# ─────────────────────────────────────────────────────────────
print("=" * 55)
print("USER FEATURES — First 3 rows")
print("=" * 55)
display(user_df.head(3))

print("\n" + "=" * 55)
print("ITEM FEATURES — First 3 rows")
print("=" * 55)
display(item_df.head(3))

print("\n" + "=" * 55)
print("INTERACTION FEATURES — First 3 rows")
print("=" * 55)
display(interaction_df.head(3))

USER FEATURES — First 3 rows


,customer_id,FN,Active,club_member_status,fashion_news_frequency,age,postal_code,purchase_count,recent_purchase_date,first_purchase_date,total_spend,avg_spend_per_txn,unique_articles,days_since_last_purchase,purchase_frequency,active_status
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0,0,ACTIVE,NONE,49,52043ee2162cf5aa7ee79974281641c6f11a68d276429a...,19.0,2020-09-05,2018-12-27,0.543932,0.028628,19.0,17.0,0.030744,1
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,0,0,ACTIVE,NONE,25,2973abc54daa8a5f8ccfe9362140c63247c5eee03f1d93...,78.0,2020-07-08,2018-09-21,2.412237,0.030926,64.0,76.0,0.118902,1
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0,0,ACTIVE,NONE,24,64f17e6a330a85798e4998f62d0930d14db8db1c054af6...,15.0,2020-09-15,2018-09-20,0.606525,0.040435,14.0,7.0,0.020661,1



ITEM FEATURES — First 3 rows


,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,...,garment_group_name,detail_desc,total_purchases,unique_buyers,avg_price,first_sold_date,last_sold_date,days_since_last_sale,popularity_score,purchases_last_30d
0,108775015,108775,Strap Top,253,Vest Top,Garment Upper Body,1010016,Solid,9,Black,...,Jersey Basic,jersey top with narrow shoulder straps.,7303.0,6885.0,0.008134,2018-09-20,2020-07-22,62.0,0.173279,0.0
1,108775044,108775,Strap Top,253,Vest Top,Garment Upper Body,1010016,Solid,10,White,...,Jersey Basic,jersey top with narrow shoulder straps.,5539.0,5284.0,0.008108,2018-09-20,2020-09-20,2.0,0.131424,11.0
2,108775051,108775,Strap Top (1),253,Vest Top,Garment Upper Body,1010017,Stripe,11,Off White,...,Jersey Basic,jersey top with narrow shoulder straps.,171.0,168.0,0.004989,2018-09-20,2019-06-28,452.0,0.004057,0.0



INTERACTION FEATURES — First 3 rows


,customer_id,article_id,t_dat,price,label,days_since_purchase,recency_weight,purchase_count_pair,is_repeat_purchase,day_of_week,month,week_of_year,year
25708980,1af0c1b1a8d547bd0761cab0a1487eced88683074fd8cc...,829501004,2020-07-05,0.042356,1,79,0.453845,1,0,6,7,27,2020
8338709,9b27546ce47806d847078a45a62ed33f9711579282f614...,708499003,2019-04-25,0.016932,1,516,0.005742,1,0,3,4,17,2019
9799464,81dd269a47400f92994ac425301cc00761b464e7b5e4df...,736626001,2019-05-28,0.027441,1,483,0.007987,1,0,1,5,22,2019


In [4]:
# ─────────────────────────────────────────────────────────────
# IDENTIFY KEY COLUMNS
# The processed interaction data uses customer_id/article_id from preprocessing.
# The retrieval model still consumes features named user_id/item_id, so we
# map the source dataframe columns here.
# ─────────────────────────────────────────────────────────────

USER_ID_COL = "customer_id"
ITEM_ID_COL = "article_id"

# Validate columns exist
assert USER_ID_COL in interaction_df.columns, f"Column '{USER_ID_COL}' not found in interaction data!"
assert ITEM_ID_COL in interaction_df.columns, f"Column '{ITEM_ID_COL}' not found in interaction data!"

# Convert IDs to strings — TFRS vocabulary layers expect string inputs
interaction_df[USER_ID_COL] = interaction_df[USER_ID_COL].astype(str)
interaction_df[ITEM_ID_COL] = interaction_df[ITEM_ID_COL].astype(str)
user_df[USER_ID_COL]        = user_df[USER_ID_COL].astype(str)
item_df[ITEM_ID_COL]        = item_df[ITEM_ID_COL].astype(str)

print(f"✅ Unique users in interactions: {interaction_df[USER_ID_COL].nunique():,}")
print(f"✅ Unique items in interactions: {interaction_df[ITEM_ID_COL].nunique():,}")
print(f"✅ Total interactions:           {len(interaction_df):,}")

✅ Unique users in interactions: 19,486
✅ Unique items in interactions: 13,464
✅ Total interactions:           20,000


## ⚡ Section 4 — Prepare TensorFlow Datasets

### Why Do We Need TensorFlow Datasets?

Pandas DataFrames are great for exploration, but for **training neural networks** we need `tf.data.Dataset` because:

1. **Batching** — feeds data in mini-batches (e.g., 128 rows at a time) instead of all at once  
2. **Shuffling** — randomizes order each epoch so the model doesn't memorize sequence  
3. **Efficiency** — streams from disk, prefetches next batch while GPU trains on current batch  
4. **Compatibility** — required by `model.fit()` in TensorFlow

In [5]:
# ─────────────────────────────────────────────────────────────
# CONFIGURATION
# Centralizing hyperparameters makes experiments easy to manage
# ─────────────────────────────────────────────────────────────
EMBEDDING_DIM  = 32    # Size of each embedding vector (32-128 is typical for retrieval)
BATCH_SIZE     = 128   # How many interactions to process per training step
EPOCHS         = 5     # How many full passes through the training data
RANDOM_SEED    = 42    # For reproducible results

tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("⚙️  Training Configuration:")
print(f"   Embedding Dimension : {EMBEDDING_DIM}")
print(f"   Batch Size          : {BATCH_SIZE}")
print(f"   Epochs              : {EPOCHS}")

⚙️  Training Configuration:
   Embedding Dimension : 32
   Batch Size          : 128
   Epochs              : 5


In [6]:
# ─────────────────────────────────────────────────────────────
# CONVERT INTERACTIONS TO TF DATASET

# Each row in interaction_df is one training example:
#   {"user_id": "U123", "item_id": "P456"}
# This tells the model: "User U123 interacted with Item P456"
# The model learns: similar embeddings = positive interaction
# ─────────────────────────────────────────────────────────────
interactions_dict = {
    "user_id": tf.constant(interaction_df[USER_ID_COL].values),
    "item_id": tf.constant(interaction_df[ITEM_ID_COL].values),
}

interactions_ds = tf.data.Dataset.from_tensor_slices(interactions_dict)

# ─────────────────────────────────────────────────────────────
# TRAIN / TEST SPLIT
# 80% training, 20% evaluation
# We shuffle first so split is random (not chronological)
# ─────────────────────────────────────────────────────────────
total_size    = len(interaction_df)
train_size    = int(total_size * 0.8)

interactions_ds = interactions_ds.shuffle(
    buffer_size=total_size,
    seed=RANDOM_SEED,
    reshuffle_each_iteration=False
)

train_ds = interactions_ds.take(train_size).batch(BATCH_SIZE).cache()
test_ds  = interactions_ds.skip(train_size).batch(BATCH_SIZE).cache()

print(f"✅ Training batches : {len(train_ds)}  (~{train_size:,} interactions)")
print(f"✅ Test batches     : {len(test_ds)}  (~{total_size - train_size:,} interactions)")

✅ Training batches : 125  (~16,000 interactions)
✅ Test batches     : 32  (~4,000 interactions)


In [7]:
# ─────────────────────────────────────────────────────────────
# BUILD ITEM CANDIDATE DATASET
# This dataset contains ALL items — the model scores every item
# against a user embedding during retrieval
# ─────────────────────────────────────────────────────────────
item_ids_ds = tf.data.Dataset.from_tensor_slices(
    tf.constant(item_df[ITEM_ID_COL].values)
)

# We batch items for efficient scoring (BruteForce layer expects batches)
items_ds = item_ids_ds.batch(BATCH_SIZE)

print(f"✅ Total candidate items: {len(item_df):,}")

✅ Total candidate items: 105,542


## 📖 Section 5 — Build User Vocabulary

### Why Build a Vocabulary?

Neural networks work with **numbers**, not strings. So `"user_123"` must be converted to an integer index, like `42`.

A **vocabulary layer** creates this mapping automatically:
```
"user_001" → 0
"user_002" → 1
"user_003" → 2
...
```
Then the embedding layer converts each integer index into a dense vector.

In [8]:
# ─────────────────────────────────────────────────────────────
# BUILD USER VOCABULARY
# StringLookup maps string IDs → integer indices
# mask_token=None → no reserved token for masking (simpler for this use case)
# ─────────────────────────────────────────────────────────────
unique_user_ids = user_df[USER_ID_COL].unique().tolist()

user_vocabulary = tf.keras.layers.StringLookup(
    vocabulary=unique_user_ids,
    mask_token=None,
    name="user_vocabulary"
)

num_users = user_vocabulary.vocabulary_size()

print(f"✅ User vocabulary size: {num_users:,} unique users")

# Quick test — see what index a user maps to
sample_user = unique_user_ids[0]
sample_index = user_vocabulary(tf.constant([sample_user]))
print(f"   Example: '{sample_user}' → index {sample_index.numpy()[0]}")

✅ User vocabulary size: 1,371,981 unique users


   Example: '00000dbacae5abe5e23885899a1fa44253a17956c6d1c3d25f88aa139fdfc657' → index 1


## 📖 Section 6 — Build Item Vocabulary

In [9]:
# ─────────────────────────────────────────────────────────────
# BUILD ITEM VOCABULARY
# Same process as user vocabulary — map item string IDs to integers
# ─────────────────────────────────────────────────────────────
unique_item_ids = item_df[ITEM_ID_COL].unique().tolist()

item_vocabulary = tf.keras.layers.StringLookup(
    vocabulary=unique_item_ids,
    mask_token=None,
    name="item_vocabulary"
)

num_items = item_vocabulary.vocabulary_size()

print(f"✅ Item vocabulary size: {num_items:,} unique items")

# Quick test
sample_item = unique_item_ids[0]
sample_index = item_vocabulary(tf.constant([sample_item]))
print(f"   Example: '{sample_item}' → index {sample_index.numpy()[0]}")

✅ Item vocabulary size: 105,543 unique items
   Example: '108775015' → index 1


## 🧬 Section 7 — Create Embedding Layers

### What is an Embedding Layer?

An **Embedding layer** is a lookup table. Given an integer index, it returns a dense vector (the embedding).

```
BEFORE TRAINING (random):
   User 0 → [0.12, -0.34, 0.88, ...] (32 numbers, random)
   User 1 → [0.55,  0.22, -0.71, ...] (32 numbers, random)

AFTER TRAINING (learned):
   User 0 (loves electronics)  → [0.91,  0.12, -0.05, ...]
   User 1 (loves fashion)      → [-0.03, 0.88,  0.77, ...]
   Item 5 (smartphone)         → [0.89,  0.10,  0.01, ...] ← similar to User 0!
   Item 9 (dress)              → [-0.02, 0.85,  0.79, ...] ← similar to User 1!
```

The model trains these random numbers into meaningful representations by learning:
> "If user clicked/bought this item → their embeddings should be similar"

In [10]:
# ─────────────────────────────────────────────────────────────
# USER EMBEDDING LAYER
# num_users = total unique users (vocabulary size)
# EMBEDDING_DIM = number of learned dimensions per user
# ─────────────────────────────────────────────────────────────
user_embedding_layer = tf.keras.layers.Embedding(
    input_dim=num_users,
    output_dim=EMBEDDING_DIM,
    name="user_embedding"
)

# ─────────────────────────────────────────────────────────────
# ITEM EMBEDDING LAYER
# ─────────────────────────────────────────────────────────────
item_embedding_layer = tf.keras.layers.Embedding(
    input_dim=num_items,
    output_dim=EMBEDDING_DIM,
    name="item_embedding"
)

print("✅ Embedding layers created:")
print(f"   User Embedding  : {num_users:,} users × {EMBEDDING_DIM} dimensions")
print(f"   Item Embedding  : {num_items:,} items × {EMBEDDING_DIM} dimensions")
print(f"   Total parameters: {(num_users + num_items) * EMBEDDING_DIM:,} learnable weights")

✅ Embedding layers created:
   User Embedding  : 1,371,981 users × 32 dimensions
   Item Embedding  : 105,543 items × 32 dimensions
   Total parameters: 47,280,768 learnable weights


## 🧠 Section 8 — Build Query Tower (User Tower)

### What Does the Query Tower Do?

The **Query Tower** (also called User Tower) takes user information and produces a **user embedding** — a numerical fingerprint of user preferences.

```
Input: user_id = "user_123"
  │
  ▼
StringLookup: "user_123" → 42
  │
  ▼  
Embedding: 42 → [0.2, 0.8, 0.1, ..., 0.5]  (32 numbers)
  │
  ▼
Output: user_embedding shape = (batch_size, 32)
```

The model learns: **users with similar preferences → similar embeddings**

In [11]:
class QueryTower(tf.keras.Model):
    """
    Query Tower (User Tower) for the Two-Tower retrieval model.

    Takes user_id strings as input and produces user embeddings.
    These embeddings encode user preferences in a vector space.

    Architecture:
        user_id (string) → StringLookup → Embedding → user_vector
    """

    def __init__(self, user_vocabulary, embedding_dim, **kwargs):
        """
        Args:
            user_vocabulary : StringLookup layer with user IDs
            embedding_dim   : Size of the output embedding vector
        """
        super().__init__(**kwargs)

        # Step 1: Map string user_id → integer index
        self.user_lookup = user_vocabulary

        # Step 2: Map integer index → dense embedding vector
        self.user_embedding = tf.keras.layers.Embedding(
            input_dim=user_vocabulary.vocabulary_size(),
            output_dim=embedding_dim,
            embeddings_initializer="glorot_uniform",  # Smart random initialization
            name="user_embedding"
        )

        # ── Optional: Dense layers for richer representations ──
        # For v1 of the model we keep it simple (no dense layers).
        # In future iterations, add:
        #   Dense(64, activation='relu') → Dense(32) for deeper understanding

    def call(self, user_ids):
        """
        Forward pass: user_ids (strings) → user embeddings.

        Args:
            user_ids : Tensor of user ID strings, shape (batch_size,)

        Returns:
            Tensor of user embeddings, shape (batch_size, embedding_dim)
        """
        # Convert string IDs to integer indices
        user_indices = self.user_lookup(user_ids)         # (batch, ) int

        # Look up embeddings for each index
        user_embeddings = self.user_embedding(user_indices)  # (batch, embedding_dim)

        return user_embeddings


# ── Instantiate the Query Tower ──
query_tower = QueryTower(
    user_vocabulary=user_vocabulary,
    embedding_dim=EMBEDDING_DIM,
    name="query_tower"
)

# ── Quick sanity check ──
test_user_batch = tf.constant([unique_user_ids[0], unique_user_ids[1]])
test_user_embedding = query_tower(test_user_batch)

print("✅ Query Tower built successfully!")
print(f"   Input  : user_ids of shape {test_user_batch.shape}")
print(f"   Output : embeddings of shape {test_user_embedding.shape}")
print(f"   Sample embedding (first 5 dims): {test_user_embedding[0, :5].numpy()}")

✅ Query Tower built successfully!
   Input  : user_ids of shape (2,)
   Output : embeddings of shape (2, 32)
   Sample embedding (first 5 dims): [-0.00085943  0.0016654  -0.00109435 -0.00052237  0.00190181]


## 🏪 Section 9 — Build Candidate Tower (Item Tower)

### What Does the Candidate Tower Do?

The **Candidate Tower** (also called Item Tower) takes item information and produces an **item embedding** — a numerical fingerprint of the item's characteristics.

```
Input: item_id = "PROD_456"
  │
  ▼
StringLookup: "PROD_456" → 17
  │
  ▼
Embedding: 17 → [0.3, 0.7, 0.2, ..., 0.4]  (32 numbers)
  │
  ▼
Output: item_embedding shape = (batch_size, 32)
```

The model learns: **items with similar content → similar embeddings**

In [12]:
class CandidateTower(tf.keras.Model):
    """
    Candidate Tower (Item Tower) for the Two-Tower retrieval model.

    Takes item_id strings as input and produces item embeddings.
    These embeddings encode item characteristics in the same vector
    space as the user embeddings — enabling similarity comparison.

    Architecture:
        item_id (string) → StringLookup → Embedding → item_vector
    """

    def __init__(self, item_vocabulary, embedding_dim, **kwargs):
        """
        Args:
            item_vocabulary : StringLookup layer with item IDs
            embedding_dim   : Size of the output embedding vector
        """
        super().__init__(**kwargs)

        # Step 1: Map string item_id → integer index
        self.item_lookup = item_vocabulary

        # Step 2: Map integer index → dense embedding vector
        self.item_embedding = tf.keras.layers.Embedding(
            input_dim=item_vocabulary.vocabulary_size(),
            output_dim=embedding_dim,
            embeddings_initializer="glorot_uniform",
            name="item_embedding"
        )

    def call(self, item_ids):
        """
        Forward pass: item_ids (strings) → item embeddings.

        Args:
            item_ids : Tensor of item ID strings, shape (batch_size,)

        Returns:
            Tensor of item embeddings, shape (batch_size, embedding_dim)
        """
        item_indices    = self.item_lookup(item_ids)
        item_embeddings = self.item_embedding(item_indices)
        return item_embeddings


# ── Instantiate the Candidate Tower ──
candidate_tower = CandidateTower(
    item_vocabulary=item_vocabulary,
    embedding_dim=EMBEDDING_DIM,
    name="candidate_tower"
)

# ── Quick sanity check ──
test_item_batch    = tf.constant([unique_item_ids[0], unique_item_ids[1]])
test_item_embedding = candidate_tower(test_item_batch)

print("✅ Candidate Tower built successfully!")
print(f"   Input  : item_ids of shape {test_item_batch.shape}")
print(f"   Output : embeddings of shape {test_item_embedding.shape}")
print(f"   Sample embedding (first 5 dims): {test_item_embedding[0, :5].numpy()}")

✅ Candidate Tower built successfully!
   Input  : item_ids of shape (2,)
   Output : embeddings of shape (2, 32)
   Sample embedding (first 5 dims): [ 0.00314851  0.00415668 -0.00233275 -0.00188528 -0.00732095]


## 🔗 Section 10 — Build the Full Retrieval Model

### How Does the Retrieval Task Work?

TensorFlow Recommenders provides a `Retrieval` task that:

1. Takes a batch of `(user_embedding, item_embedding)` pairs  
2. Computes **dot products** between ALL pairs in the batch  
3. Treats the "correct" pairs (from actual interactions) as positives  
4. Treats all other pairs as **in-batch negatives** (a clever trick!)  
5. Uses **Categorical Cross-Entropy** to maximize scores for positive pairs

```
Batch of 4 interactions:
  (User A, Item 1)  ← positive pair
  (User B, Item 5)  ← positive pair
  (User C, Item 3)  ← positive pair
  (User D, Item 7)  ← positive pair

The model also implicitly sees:
  (User A, Item 5)  ← negative pair (A didn't click Item 5)
  (User A, Item 3)  ← negative pair
  ... and so on for all cross-pairs

Goal: Score(User A, Item 1) >> Score(User A, Item 5 or 3 or 7)
```

This technique is called **In-Batch Negative Sampling** and is very efficient!

In [13]:
# ─────────────────────────────────────────────────────────────
# BUILD ITEM CANDIDATE DATASET
# This dataset contains ALL items — the model scores every item
# against a user embedding during retrieval
# ─────────────────────────────────────────────────────────────
item_ids_ds = tf.data.Dataset.from_tensor_slices(
    tf.constant(item_df[ITEM_ID_COL].values)
)

# We batch items for efficient scoring (BruteForce layer expects batches)
items_ds = item_ids_ds.batch(BATCH_SIZE)

print(f"✅ Total candidate items: {len(item_df):,}")


class TwoTowerRetrievalModel(tfrs.models.Model):
    """
    Full Two-Tower Retrieval Model.

    Combines the QueryTower and CandidateTower with a TFRS Retrieval
    task to learn user and item embeddings simultaneously.

    Training Objective:
        Maximize similarity between embeddings of users and items
        they actually interacted with, while minimizing similarity
        for non-interacted pairs (in-batch negatives).
    """

    def __init__(self, query_tower, candidate_tower, items_dataset):
        """
        Args:
            query_tower     : QueryTower instance (user encoder)
            candidate_tower : CandidateTower instance (item encoder)
            items_dataset   : tf.data.Dataset of all item IDs (for evaluation)
        """
        super().__init__()

        # The two towers
        self.query_tower     = query_tower
        self.candidate_tower = candidate_tower

        # ── RETRIEVAL TASK ──
        # This defines the loss function and evaluation metric (Factorized Top-K)
        # metrics: how often the true item is in the Top-K recommended items
        self.retrieval_task = tfrs.tasks.Retrieval(
            metrics=tfrs.metrics.FactorizedTopK(
                candidates=items_dataset.map(candidate_tower)
            )
        )

    def compute_loss(self, features, training=False):
        """
        Called automatically during model.fit().

        Takes a batch of interactions → computes retrieval loss.

        Args:
            features : dict with keys 'user_id' and 'item_id'
            training : bool, whether in training mode

        Returns:
            Scalar loss value to minimize
        """
        # Step 1: Get user embeddings from the query tower
        user_embeddings = self.query_tower(features["user_id"])

        # Step 2: Get item embeddings from the candidate tower
        item_embeddings = self.candidate_tower(features["item_id"])

        # Step 3: Compute retrieval loss (in-batch negatives)
        loss = self.retrieval_task(
            query_embeddings=user_embeddings,
            candidate_embeddings=item_embeddings,
            compute_metrics=not training  # Metrics only on eval (faster training)
        )

        return loss


# ── Instantiate the full model ──
model = TwoTowerRetrievalModel(
    query_tower=query_tower,
    candidate_tower=candidate_tower,
    items_dataset=items_ds
)

print("✅ Two-Tower Retrieval Model assembled!")
print("   Components:")
print("   ├── QueryTower     (user encoder)")
print("   ├── CandidateTower (item encoder)")
print("   └── Retrieval Task (in-batch negatives + FactorizedTopK metrics)")

✅ Total candidate items: 105,542
✅ Two-Tower Retrieval Model assembled!
   Components:
   ├── QueryTower     (user encoder)
   ├── CandidateTower (item encoder)
   └── Retrieval Task (in-batch negatives + FactorizedTopK metrics)


## 🏋️ Section 11 — Train the Model

In [14]:
# ─────────────────────────────────────────────────────────────
# COMPILE
# Adam optimizer is the standard choice for recommendation models
# learning_rate=0.1 is higher than typical — TFRS retrieval benefits from this
# ─────────────────────────────────────────────────────────────
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.1)
)

print("⚙️  Model compiled with Adam optimizer (lr=0.1)")
print(f"\n🏋️  Starting training for {EPOCHS} epochs...")
print("=" * 60)

⚙️  Model compiled with Adam optimizer (lr=0.1)

🏋️  Starting training for 5 epochs...


In [15]:
# ─────────────────────────────────────────────────────────────
# TRAIN
# model.fit() iterates over train_ds for EPOCHS passes
# After each epoch, it evaluates on test_ds

# READING THE METRICS:
#   loss                     → lower is better (retrieval objective)
#   factorized_top_k/top_1_categorical_accuracy  → % correct in Top-1
#   factorized_top_k/top_5_categorical_accuracy  → % correct in Top-5
#   factorized_top_k/top_10_categorical_accuracy → % correct in Top-10
#   factorized_top_k/top_50_categorical_accuracy → % correct in Top-50
#   factorized_top_k/top_100_categorical_accuracy→ % correct in Top-100
# ─────────────────────────────────────────────────────────────

history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=EPOCHS,
    verbose=1
)

print("\n✅ Training complete!")

Epoch 1/5
125/125 [==============================] - 138s 969ms/step - factorized_top_k/top_1_categorical_accuracy: 0.0000e+00 - factorized_top_k/top_5_categorical_accuracy: 0.0000e+00 - factorized_top_k/top_10_categorical_accuracy: 0.0000e+00 - factorized_top_k/top_50_categorical_accuracy: 0.0000e+00 - factorized_top_k/top_100_categorical_accuracy: 0.0000e+00 - loss: 628.2797 - regularization_loss: 0.0000e+00 - total_loss: 628.2797 - val_factorized_top_k/top_1_categorical_accuracy: 0.0000e+00 - val_factorized_top_k/top_5_categorical_accuracy: 5.0000e-04 - val_factorized_top_k/top_10_categorical_accuracy: 0.0010 - val_factorized_top_k/top_50_categorical_accuracy: 0.0050 - val_factorized_top_k/top_100_categorical_accuracy: 0.0072 - val_loss: 119.4095 - val_regularization_loss: 0.0000e+00 - val_total_loss: 119.4095
Epoch 2/5
125/125 [==============================] - 120s 957ms/step - factorized_top_k/top_1_categorical_accuracy: 0.0000e+00 - factorized_top_k/top_5_categorical_accuracy: 0

In [16]:
# ─────────────────────────────────────────────────────────────
# EVALUATE ON TEST SET
# ─────────────────────────────────────────────────────────────
print("📊 Final Evaluation on Test Set:")
print("=" * 50)

eval_results = model.evaluate(test_ds, return_dict=True, verbose=0)

for metric_name, metric_value in eval_results.items():
    print(f"   {metric_name:<52}: {metric_value:.4f}")

print("\n💡 How to read these results:")
print("   Top-10 accuracy of 0.80 means: for 80% of test users,")
print("   the item they actually interacted with appears in our Top-10 recommendations.")

📊 Final Evaluation on Test Set:
   factorized_top_k/top_1_categorical_accuracy         : 0.0000
   factorized_top_k/top_5_categorical_accuracy         : 0.0003
   factorized_top_k/top_10_categorical_accuracy        : 0.0003
   factorized_top_k/top_50_categorical_accuracy        : 0.0018
   factorized_top_k/top_100_categorical_accuracy       : 0.0027
   loss                                                : 152.4205
   regularization_loss                                 : 0.0000
   total_loss                                          : 152.4205

💡 How to read these results:
   Top-10 accuracy of 0.80 means: for 80% of test users,
   the item they actually interacted with appears in our Top-10 recommendations.


## 🎯 Section 12 — Generate Recommendations

### How Recommendation Retrieval Works at Inference Time

```
1. Compute user embedding for target user
2. Compute ALL item embeddings (once, cached)
3. Find the K items with highest similarity to the user embedding
4. Return those K items as recommendations

Real systems use ANN (Approximate Nearest Neighbor) search (e.g., FAISS/ScaNN)
for this step when item catalogs have millions of items.
For now we use TFRS BruteForce — exact search, fine for smaller catalogs.
```

In [25]:
# ─────────────────────────────────────────────────────────────
# BUILD A SMALLER RECOMMENDATION CHECK INDEX
# For local experimentation we use a sampled pool of items instead of the
# entire catalog. Full catalogs are what production systems use, but they can
# be expensive to query on a laptop. A smaller pool lets us quickly validate
# that the two-tower architecture is producing sensible top-K results.
# ─────────────────────────────────────────────────────────────
print("⏳ Building quick recommendation index from a sampled item pool...")

sampled_item_ids_ds = item_ids_ds.take(1000)
audit_index = tfrs.layers.factorized_top_k.BruteForce(
    query_model=model.query_tower,
    k=10
)

audit_index.index_from_dataset(
    tf.data.Dataset.zip((
        sampled_item_ids_ds,
        sampled_item_ids_ds.map(model.candidate_tower)
    )).batch(BATCH_SIZE)
)

print("✅ Quick recommendation index built with 1,000 sampled items")

⏳ Building quick recommendation index from a sampled item pool...
✅ Quick recommendation index built with 1,000 sampled items


In [26]:
# ─────────────────────────────────────────────────────────────
# GENERATE TOP-10 RECOMMENDATIONS FOR SAMPLE USERS
# Models usually return article IDs because IDs are the stable keys used
# for training, storage, and retrieval. We map those IDs back to product
# metadata so the notebook shows human-friendly results.
# Metadata makes the recommendations easier to understand during
# experimentation and presentation.
# ─────────────────────────────────────────────────────────────

RECOMMENDATION_COLUMNS = [
    "article_id",
    "product_type_name",
    "colour_group_name",
    "garment_group_name",
    "product_group_name",
    "detail_desc",
]

articles_lookup_df = articles_df[RECOMMENDATION_COLUMNS].drop_duplicates("article_id").copy()


def get_recommendations(user_id: str, top_k: int = 10) -> list:
    """
    Generate Top-K product recommendations for a given user.

    Args:
        user_id : The user ID string (must exist in vocabulary)
        top_k   : Number of recommendations to return

    Returns:
        List of recommended item ID strings, ordered by relevance score
    """
    # Reshape to (1,) — model expects a batch dimension
    user_tensor = tf.constant([user_id])

    # Query the sampled audit index
    scores, item_ids = audit_index(user_tensor, k=top_k)

    # Convert bytes tensor to Python list of strings
    recommended_items = [item_id.decode("utf-8") for item_id in item_ids[0].numpy()]
    recommendation_scores = scores[0].numpy().tolist()

    return list(zip(recommended_items, recommendation_scores))


def build_readable_recommendations(user_id: str, top_k: int = 10) -> pd.DataFrame:
    """
    Convert model output IDs into a readable recommendation table.

    Why this matters:
    - recommendation models return IDs because IDs are the reliable join key
    - product metadata helps us explain the result to humans
    - this is the same idea used in real recommendation dashboards
    """
    recs = get_recommendations(user_id, top_k=top_k)
    recs_df = pd.DataFrame(recs, columns=["article_id", "score"])
    recs_df["article_id"] = recs_df["article_id"].astype(str)

    readable_df = recs_df.merge(
        articles_lookup_df,
        on="article_id",
        how="left"
    )

    readable_df = readable_df.fillna({
        "product_type_name": "Unknown",
        "colour_group_name": "Unknown",
        "garment_group_name": "Unknown",
        "product_group_name": "Unknown",
        "detail_desc": "No description available",
    })

    readable_df.insert(0, "rank", range(1, len(readable_df) + 1))
    readable_df["detail_desc"] = readable_df["detail_desc"].astype(str).str.slice(0, 120)

    readable_df = readable_df.rename(columns={
        "article_id": "Product ID",
        "product_type_name": "Product Type",
        "colour_group_name": "Color",
        "garment_group_name": "Garment Group",
        "product_group_name": "Category",
        "detail_desc": "Description",
        "score": "Score",
    })

    return readable_df[[
        "rank",
        "Product ID",
        "Product Type",
        "Color",
        "Garment Group",
        "Category",
        "Score",
        "Description",
    ]]


# ── Generate readable recommendations for 3 sample users ──
sample_users = unique_user_ids[:3]

print("🎯 Readable Top-10 Recommendations (sampled candidate pool):\n")

for user_id in sample_users:
    readable_recs = build_readable_recommendations(user_id, top_k=10)
    print(f"Recommended Products for User {user_id}:")
    display(readable_recs.style.format({"Score": "{:.4f}"}))
    print()

🎯 Readable Top-10 Recommendations (sampled candidate pool):

Recommended Products for User 00000dbacae5abe5e23885899a1fa44253a17956c6d1c3d25f88aa139fdfc657:


,rank,Product ID,Product Type,Color,Garment Group,Category,Score,Description
0,1,224606019,Belt,Black,Accessories,Accessories,0.0483,narrow belt in grained imitation leather with a metal buckle. width 2 cm.
1,2,188183001,Swimsuit,Black,Swimwear,Swimwear,0.0464,"fully lined shaping swimsuit that has a sculpting effect on the tummy, back and bum. wrapover top, lightly padded cups,"
2,3,291333012,Underwear Tights,Black,Socks And Tights,Socks & Tights,0.0301,"tights in a soft, fine-knit cotton blend with an elasticated waist."
3,4,228257002,Underwear Tights,Beige,Socks And Tights,Socks & Tights,0.0300,tights with an elasticated waist. 20 denier.
4,5,201219014,Underwear Tights,Dark Red,Socks And Tights,Socks & Tights,0.0287,fine-knit tights with an elasticated waist.
5,6,200761028,Vest Top,Dark Blue,Jersey Basic,Garment Upper Body,0.0284,"vest tops in soft, ribbed organic cotton jersey."
6,7,307239004,Underwear Bottom,Light Beige,"Under-, Nightwear",Underwear,0.0284,"shaping briefs in microfibre with a high waist, laser-cut edges and lined gusset. the briefs have a light shaping effect"
7,8,189616001,Leggings/Tights,Black,Jersey Basic,Garment Lower Body,0.0281,leggings in extra sturdy jersey with an elasticated waist.
8,9,252298002,Trousers,Black,Dresses Ladies,Garment Lower Body,0.0276,"jeans in washed, stretch denim with hard-worn details, a regular waist, front and back pockets and skinny legs."
9,10,237222012,Top,White,Jersey Basic,Garment Upper Body,0.0275,"fitted henley top in soft cotton jersey with a low-cut neckline, button placket and long sleeves."



Recommended Products for User 0000423b00ade91418cceaf3b26c6af3dd342b51fd051eec9c12fb36984420fa:


,rank,Product ID,Product Type,Color,Garment Group,Category,Score,Description
0,1,156224002,Unknown,Black,Socks And Tights,Unknown,0.0392,semi-matte socks with a short shaft. 20 denier.
1,2,158340001,Leggings/Tights,Black,Socks And Tights,Garment Lower Body,0.0292,high-waisted tights that lift the bum and shape the waist and thighs. 30 denier.
2,3,305775041,Underwear Tights,Light Pink,Socks And Tights,Socks & Tights,0.0270,tights in a fine knit containing glittery threads with an elasticated waist. soft inside.
3,4,160442043,Socks,Light Grey,Socks And Tights,Socks & Tights,0.0241,"short, fine-knit socks designed to be hidden by your shoes with a silicone trim at the back of the heel to keep them in"
4,5,108775044,Vest Top,White,Jersey Basic,Garment Upper Body,0.0231,jersey top with narrow shoulder straps.
5,6,294008002,Costumes,Black,Jersey Fancy,Garment Full Body,0.0222,gently flared top in soft viscose jersey with short sleeves and a rounded hem.
6,7,189654046,Skirt,Red,Jersey Basic,Garment Lower Body,0.0221,short jersey skirt with an elasticated waist.
7,8,224314013,Hat/Beanie,Beige,Accessories,Accessories,0.0220,cable-knit hat with a faux fur pompom.
8,9,212629004,Dress,Black,Jersey Basic,Garment Full Body,0.0219,"long, sleeveless dress in jersey with narrow shoulder straps and an elasticated seam at the waist. unlined."
9,10,252298002,Trousers,Black,Dresses Ladies,Garment Lower Body,0.0218,"jeans in washed, stretch denim with hard-worn details, a regular waist, front and back pockets and skinny legs."



Recommended Products for User 000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318:


,rank,Product ID,Product Type,Color,Garment Group,Category,Score,Description
0,1,153115040,Bra,Yellowish Brown,"Under-, Nightwear",Underwear,0.0336,"strapless bra in microfibre with underwired, padded cups that lift and shape the bust. silicone trim at the top and a ho"
1,2,179208001,Leggings/Tights,Black,Socks And Tights,Garment Lower Body,0.0291,matt opaque tights with a control top to hold in the tummy and bum. 100 denier.
2,3,189616014,Leggings/Tights,Black,Jersey Basic,Garment Lower Body,0.0284,leggings in extra sturdy jersey with an elasticated waist.
3,4,160442043,Socks,Light Grey,Socks And Tights,Socks & Tights,0.0267,"short, fine-knit socks designed to be hidden by your shoes with a silicone trim at the back of the heel to keep them in"
4,5,146730001,Leggings/Tights,Black,Socks And Tights,Garment Lower Body,0.0259,opaque matt leggings with an elasticated waist. 200 denier.
5,6,240561001,Underwear Tights,Black,Socks And Tights,Socks & Tights,0.0222,"semi shiny tights that shape the tummy, thighs and calves, while also encouraging blood circulation in the legs. elastic"
6,7,200182002,Underwear Tights,Black,Socks And Tights,Socks & Tights,0.0213,tights with an elasticated waist. 40 denier.
7,8,156224002,Unknown,Black,Socks And Tights,Unknown,0.0204,semi-matte socks with a short shaft. 20 denier.
8,9,148033001,Leggings/Tights,Black,Socks And Tights,Garment Lower Body,0.0196,semi shiny stay-ups with a wide lace trim at the top and silicone on the inside. 20 denier.
9,10,283236034,Shirt,Beige,Shirts,Garment Upper Body,0.0196,"shirt in a cotton weave with a button-down collar, classic front and yoke at the back. open chest pocket, short sleeves"


## 🔬 Section 13 — Understanding Embeddings

Let's inspect the learned embeddings to build intuition about what the model discovered.

In [27]:
# ─────────────────────────────────────────────────────────────
# INSPECT LEARNED USER EMBEDDINGS
# ─────────────────────────────────────────────────────────────
print("🔬 Learned User Embeddings")
print("=" * 55)

for user_id in unique_user_ids[:3]:
    embedding = model.query_tower(tf.constant([user_id]))
    emb_np = embedding[0].numpy()
    print(f"\n👤 {user_id}")
    print(f"   Embedding (first 8 dims): {emb_np[:8].round(3)}")
    print(f"   Vector norm: {np.linalg.norm(emb_np):.4f}")

print("\n" + "=" * 55)
print("💡 What these numbers mean:")
print("   • Each dimension captures a latent (hidden) concept")
print("   • The model decides WHAT each dimension means")
print("   • Similar users will have similar embedding values")
print("   • We can use cosine similarity to find 'user neighbors'")

🔬 Learned User Embeddings

👤 00000dbacae5abe5e23885899a1fa44253a17956c6d1c3d25f88aa139fdfc657
   Embedding (first 8 dims): [-0.001  0.002 -0.001 -0.001  0.002  0.001 -0.002 -0.001]
   Vector norm: 0.0072

👤 0000423b00ade91418cceaf3b26c6af3dd342b51fd051eec9c12fb36984420fa
   Embedding (first 8 dims): [ 0.002 -0.001 -0.001 -0.    -0.002  0.001  0.002  0.001]
   Vector norm: 0.0070

👤 000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318
   Embedding (first 8 dims): [-0.001 -0.001 -0.001  0.001  0.002  0.002  0.     0.002]
   Vector norm: 0.0061

💡 What these numbers mean:
   • Each dimension captures a latent (hidden) concept
   • The model decides WHAT each dimension means
   • Similar users will have similar embedding values
   • We can use cosine similarity to find 'user neighbors'


In [28]:
# ─────────────────────────────────────────────────────────────
# COMPUTE USER-USER SIMILARITY
# Users with high cosine similarity → similar preferences
# This is the foundation of Collaborative Filtering
# ─────────────────────────────────────────────────────────────

def cosine_similarity(vec_a, vec_b):
    """Compute cosine similarity between two vectors. Range: [-1, 1]"""
    dot = np.dot(vec_a, vec_b)
    norm_a = np.linalg.norm(vec_a)
    norm_b = np.linalg.norm(vec_b)
    return dot / (norm_a * norm_b + 1e-9)  # +epsilon to avoid div by zero


# Get embeddings for first 5 users
sample_user_ids = unique_user_ids[:5]
embeddings = [
    model.query_tower(tf.constant([uid]))[0].numpy()
    for uid in sample_user_ids
]

# Build similarity matrix
print("👥 User-User Similarity Matrix (Cosine Similarity)")
print("   1.0 = identical preferences | 0.0 = unrelated | -1.0 = opposite")
print()

sim_df = pd.DataFrame(
    [[cosine_similarity(embeddings[i], embeddings[j]) for j in range(5)]
     for i in range(5)],
    index=sample_user_ids,
    columns=sample_user_ids
).round(4)

display(sim_df)

👥 User-User Similarity Matrix (Cosine Similarity)
   1.0 = identical preferences | 0.0 = unrelated | -1.0 = opposite



,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3d25f88aa139fdfc657,0000423b00ade91418cceaf3b26c6af3dd342b51fd051eec9c12fb36984420fa,000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2c5feb1ca5dff07c43e,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801fe7fc0f26dd8d65a85a
00000dbacae5abe5e23885899a1fa44253a17956c6d1c3d25f88aa139fdfc657,1.0000,-0.3781,-0.1700,-0.0112,-0.0543
0000423b00ade91418cceaf3b26c6af3dd342b51fd051eec9c12fb36984420fa,-0.3781,1.0000,0.1084,0.0558,-0.2077
000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318,-0.1700,0.1084,1.0000,-0.1609,-0.0043
00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2c5feb1ca5dff07c43e,-0.0112,0.0558,-0.1609,1.0000,0.3082
00006413d8573cd20ed7128e53b7b13819fe5cfc2d801fe7fc0f26dd8d65a85a,-0.0543,-0.2077,-0.0043,0.3082,1.0000


## 🏆 Section 14 — Final Conclusions

### ✅ What Did We Achieve in This Notebook?

| Step | What We Did | Why It Matters |
|---|---|---|
| Data Prep | Converted DataFrames → TF Datasets | Required for efficient neural network training |
| Vocabulary | Built StringLookup layers for users & items | Converts string IDs to model-compatible integers |
| Embeddings | Created 32-dimensional vector representations | Captures latent preferences/characteristics |
| Query Tower | Neural network that encodes users | Learns: what does this user like? |
| Candidate Tower | Neural network that encodes items | Learns: what type of users like this item? |
| Retrieval Model | Combined both towers with TFRS task | Trains embeddings via in-batch negatives |
| Recommendations | BruteForce Top-K retrieval | Returns ranked items for any user |

### 🌍 How Does This Connect to Real Systems?

```
Netflix / YouTube / Amazon
       │
       ├── RETRIEVAL (what we built)
       │      └── Two-Tower model → Top-500 candidates
       │
       ├── RANKING (next notebook)
       │      └── Deep & Cross Network → Top-10 ordered list
       │
       └── SERVING (future notebooks)
              └── ANN index (FAISS/ScaNN) + Redis cache + API
```

### 🚀 What's Next?

1. **Notebook 05** — Ranking Model (Deep & Cross Network)
2. **Notebook 06** — Contextual Features (time, device, session)
3. **Notebook 07** — Model Evaluation & A/B Testing Framework
4. **Notebook 08** — Production Serving Pipeline